In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. The Client-Side Model (Bottom Model)
class BottomModel(nn.Module):
    def __init__(self, input_features, embedding_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.ReLU(),
            nn.Linear(16, embedding_dim)
        )
    def forward(self, x):
        return self.net(x)

# 2. The Server-Side Model (Top Model)
class TopServerModel(nn.Module):
    def __init__(self, combined_embedding_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(combined_embedding_dim, 8),
            nn.ReLU(),
            nn.Linear(8, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# ==========================================
# EXECUTION SCRIPT
# ==========================================
if __name__ == "__main__":
    print("--- Starting Assignment 6 Simulation (Vertical FL) ---\n")

    # Setup Data for 100 Shared Users
    num_users = 100

    # Client A has 10 features (e.g., banking data)
    data_A = torch.randn(num_users, 10)
    # Client B has 5 different features (e.g., e-commerce data)
    data_B = torch.randn(num_users, 5)

    # The true labels (e.g., will this user default on a loan?)
    # Usually held by the Server or one of the active clients
    targets = torch.randint(0, 2, (num_users,))

    # Initialize Models
    client_A_model = BottomModel(input_features=10, embedding_dim=8)
    client_B_model = BottomModel(input_features=5, embedding_dim=8)
    # Server takes both 8-dim embeddings (8 + 8 = 16)
    server_model = TopServerModel(combined_embedding_dim=16, num_classes=2)

    # Initialize Optimizers
    opt_A = optim.Adam(client_A_model.parameters(), lr=0.01)
    opt_B = optim.Adam(client_B_model.parameters(), lr=0.01)
    opt_server = optim.Adam(server_model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    print("Beginning Joint Training across Clients and Server...\n")

    # Training Loop for 10 Epochs
    epochs = 10
    for epoch in range(epochs):
        # 1. Zero out previous gradients
        opt_A.zero_grad()
        opt_B.zero_grad()
        opt_server.zero_grad()

        # 2. Forward pass on clients (Generate Embeddings)
        embed_A = client_A_model(data_A)
        embed_B = client_B_model(data_B)

        # 3. Server concatenates the embeddings
        combined_embed = torch.cat((embed_A, embed_B), dim=1)

        # 4. Server computes prediction and loss
        output = server_model(combined_embed)
        loss = criterion(output, targets)

        # 5. Backward pass (Calculates gradients all the way back to the clients)
        loss.backward()

        # 6. Update weights
        opt_server.step()
        opt_A.step()
        opt_B.step()

        print(f"Epoch {epoch+1}/{epochs} | Joint Loss: {loss.item():.4f}")

    print("\nTraining Complete! Notice how the loss decreases, proving the split models are learning together without sharing raw features.")

--- Starting Assignment 6 Simulation (Vertical FL) ---

Beginning Joint Training across Clients and Server...

Epoch 1/10 | Joint Loss: 0.6849
Epoch 2/10 | Joint Loss: 0.6796
Epoch 3/10 | Joint Loss: 0.6745
Epoch 4/10 | Joint Loss: 0.6684
Epoch 5/10 | Joint Loss: 0.6610
Epoch 6/10 | Joint Loss: 0.6523
Epoch 7/10 | Joint Loss: 0.6424
Epoch 8/10 | Joint Loss: 0.6319
Epoch 9/10 | Joint Loss: 0.6207
Epoch 10/10 | Joint Loss: 0.6092

Training Complete! Notice how the loss decreases, proving the split models are learning together without sharing raw features.
